# FullStack IA · Analyse exploratoire des données

**Objectif :** NumPy, pandas, visualisation et statistiques.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## NumPy : calculer sur des tableaux

**NumPy** est la base de tout le calcul scientifique en Python : pandas, scikit-learn et PyTorch reposent dessus. Son objet central est le **tableau** (`ndarray`) : une grille de nombres du même type, sur laquelle les calculs sont très rapides.

In [ ]:
import numpy as np

a = np.array([3.0, 8.0, 1.0, 6.0])
print(a * 2)          # l'opération s'applique à chaque élément
print(a + 10)
print(a.mean(), a.std(), a.max())

Ce calcul **élément par élément** sans boucle s'appelle la **vectorisation**. C'est à la fois plus court et des dizaines de fois plus rapide qu'une boucle `for`.

## Forme et dimensions

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.shape)        # (2, 3) : 2 lignes, 3 colonnes
print(M.ndim, M.dtype)
print(M[0, 2])        # ligne 0, colonne 2
print(M[:, 1])        # toute la colonne 1
print(M.sum(axis=0))  # somme de chaque colonne
print(M.sum(axis=1))  # somme de chaque ligne

`axis=0` agrège **le long des lignes** (un résultat par colonne), `axis=1` le long des colonnes (un résultat par ligne). En ML, une ligne est un exemple et une colonne une caractéristique.

## Créer des tableaux

In [ ]:
print(np.zeros(3), np.ones((2, 2)))
print(np.arange(0, 10, 2))          # comme range
print(np.linspace(0, 1, 5))         # 5 points régulièrement espacés
rng = np.random.default_rng(42)     # générateur aléatoire reproductible
print(rng.normal(0, 1, size=3))

## Les masques booléens

Une comparaison renvoie un tableau de `True`/`False`, qu'on peut utiliser pour **filtrer**. C'est exactement ce que tu feras avec pandas.

In [ ]:
notes = np.array([12, 7, 15, 9, 18])
masque = notes >= 10
print(masque)
print(notes[masque])              # les notes >= 10
print(notes[(notes > 8) & (notes < 16)])   # & = et, | = ou, avec des parenthèses
print(masque.mean())              # proportion de True

## Le broadcasting

NumPy sait combiner des tableaux de formes différentes en « étirant » le plus petit. Exemple : centrer chaque colonne d'une matrice en soustrayant sa moyenne.

In [ ]:
X = np.array([[1.0, 200.0],
              [3.0, 400.0],
              [5.0, 600.0]])
moyennes = X.mean(axis=0)     # forme (2,)
X_centre = X - moyennes       # (3, 2) - (2,) : la ligne est répétée
X_centre

> **À retenir** : pense « tableau entier » plutôt que « boucle ». Si tu écris une boucle `for` sur les éléments d'un tableau NumPy, il existe presque toujours une version vectorisée.

### Exercice 1 · Normalisation min-max vectorisée

Sans boucle, crée `x_norm` : les valeurs de `x` ramenées entre 0 et 1 avec `(x - min) / (max - min)`.

<details><summary>Indice</summary>

x.min() et x.max() renvoient des nombres ; l'opération s'applique ensuite à tout le tableau.
</details>

In [ ]:
import numpy as np
x = np.array([4.0, 10.0, 7.0, 1.0, 13.0])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert isinstance(x_norm, np.ndarray), "x_norm doit être un tableau NumPy."
assert np.allclose(x_norm, [0.25, 0.75, 0.5, 0.0, 1.0]), f"x_norm vaut {x_norm}."
print("Bravo, tout est correct.")

### Exercice 2 · Filtrer avec un masque

Le tableau `latences` contient des temps de réponse d'une API en millisecondes. Crée `lentes`, les latences strictement supérieures à 200, et `part_lentes`, leur proportion (entre 0 et 1).

<details><summary>Indice</summary>

latences > 200 est un masque. La moyenne d'un masque donne la proportion de True.
</details>

In [ ]:
import numpy as np
latences = np.array([120, 340, 95, 210, 180, 600, 150, 205])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert list(lentes) == [340, 210, 600, 205], f"lentes vaut {lentes}."
assert abs(part_lentes - 0.5) < 1e-9, f"part_lentes vaut {part_lentes}."
print("Bravo, tout est correct.")

### Exercice 3 · Standardiser les colonnes

Standardise chaque **colonne** de `X` : soustrais sa moyenne et divise par son écart-type (`axis=0`). Stocke le résultat dans `X_std`. Chaque colonne de `X_std` doit avoir une moyenne de 0 et un écart-type de 1.

<details><summary>Indice</summary>

X.mean(axis=0) et X.std(axis=0) donnent une valeur par colonne ; le broadcasting fait le reste.
</details>

In [ ]:
import numpy as np
X = np.array([[170, 65], [180, 80], [160, 55], [175, 72]], dtype=float)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert X_std.shape == (4, 2), f"X_std a la forme {X_std.shape}, j'attendais (4, 2)."
assert np.allclose(X_std.mean(axis=0), 0) and np.allclose(X_std.std(axis=0), 1), "Chaque colonne doit avoir une moyenne 0 et un écart-type 1 : as-tu bien utilisé axis=0 ?"
print("Bravo, tout est correct.")

---
## pandas : charger et explorer

**pandas** manipule des tableaux de données avec des colonnes nommées : les **DataFrame**. Chaque colonne est une **Series**. C'est l'outil que tu utiliseras le plus pendant la formation.

In [ ]:
import pandas as pd

df = pd.read_csv("penguins.csv")
df.head()

## Les premiers réflexes sur un nouveau jeu de données

In [ ]:
print(df.shape)          # (lignes, colonnes)
print(df.columns.tolist())
df.info()                # types et valeurs non nulles par colonne

In [ ]:
df.describe()            # statistiques des colonnes numériques

In [ ]:
df["species"].value_counts()    # effectifs par catégorie

## Sélectionner des colonnes et des lignes

In [ ]:
df["body_mass_g"]                      # une colonne : une Series
df[["species", "body_mass_g"]]         # plusieurs colonnes : un DataFrame

`loc` sélectionne par **étiquettes** (noms de lignes et de colonnes), `iloc` par **positions** numériques.

In [ ]:
print(df.loc[0, "species"])            # ligne d'étiquette 0, colonne species
print(df.iloc[0, 0])                   # première ligne, première colonne
df.loc[:4, ["island", "sex"]]          # avec loc, la borne de fin est incluse

## Filtrer avec des conditions

Même principe que les masques NumPy. Les parenthèses sont obligatoires autour de chaque condition.

In [ ]:
lourds = df[df["body_mass_g"] > 5500]
print(len(lourds), "manchots de plus de 5,5 kg")
gentoo_femelles = df[(df["species"] == "Gentoo") & (df["sex"] == "FEMALE")]
gentoo_femelles.head(3)

In [ ]:
df[df["island"].isin(["Dream", "Biscoe"])].shape

## Trier

In [ ]:
df.sort_values("flipper_length_mm", ascending=False).head(3)

> **À retenir** : `df[...]` avec une **chaîne** sélectionne une colonne ; avec un **masque booléen**, il filtre des lignes. Pour combiner des conditions, utilise `&` et `|` (pas `and`/`or`), avec des parenthèses.

### Exercice 1 · Premiers chiffres

Charge `penguins.csv` dans `df`. Crée `n_lignes` (nombre de lignes), `n_gentoo` (nombre de manchots de l'espèce Gentoo) et `ile_max` (le nom de l'île qui compte le plus de manchots).

<details><summary>Indice</summary>

len(df) ou df.shape[0]. value_counts() trie par effectif ; .idxmax() ou .index[0] donne la catégorie la plus fréquente.
</details>

In [ ]:
import pandas as pd

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert n_lignes == 344, f"n_lignes vaut {n_lignes}."
assert n_gentoo == 124, f"n_gentoo vaut {n_gentoo}."
assert ile_max == "Biscoe", f"ile_max vaut {ile_max!r}."
print("Bravo, tout est correct.")

### Exercice 2 · Filtre combiné

Crée `selection` : les manchots de l'île `Dream` dont le bec (`bill_length_mm`) mesure plus de 45 mm, avec seulement les colonnes `species`, `bill_length_mm` et `sex`, triés par longueur de bec décroissante.

<details><summary>Indice</summary>

Filtre d'abord avec deux conditions et &, puis sélectionne les colonnes avec [[...]], puis sort_values(..., ascending=False).
</details>

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
ref = pd.read_csv("penguins.csv")
ref = ref[(ref.island == "Dream") & (ref.bill_length_mm > 45)][["species", "bill_length_mm", "sex"]].sort_values("bill_length_mm", ascending=False)
assert list(selection.columns) == ["species", "bill_length_mm", "sex"], f"Colonnes : {list(selection.columns)}."
assert len(selection) == len(ref), f"selection a {len(selection)} lignes, j'en attendais {len(ref)}."
assert list(selection["bill_length_mm"]) == list(ref["bill_length_mm"]), "L'ordre n'est pas le bon : trie par bill_length_mm décroissant."
print("Bravo, tout est correct.")

---
## Nettoyer les données

En pratique, le nettoyage prend une grande partie du temps d'un projet data. Un modèle entraîné sur des données sales donne des résultats faux, sans forcément lever d'erreur.

## Les valeurs manquantes

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv")
df.isna().sum()          # nombre de valeurs manquantes par colonne

Trois stratégies, selon le contexte :

In [ ]:
sans_na = df.dropna()                                     # 1. supprimer les lignes incomplètes
print(len(df), "→", len(sans_na))

df_imp = df.copy()
mediane = df_imp["body_mass_g"].median()
df_imp["body_mass_g"] = df_imp["body_mass_g"].fillna(mediane)   # 2. imputer (médiane, moyenne...)

df_sub = df.dropna(subset=["sex"])                        # 3. supprimer seulement si une colonne précise manque
print(df_imp["body_mass_g"].isna().sum(), len(df_sub))

La **médiane** est souvent préférée à la moyenne pour imputer, car elle est moins sensible aux valeurs extrêmes.

## Les doublons

In [ ]:
doublons = pd.concat([df.head(3), df.head(3)])
print(doublons.duplicated().sum())
doublons.drop_duplicates()

## Types, noms de colonnes et texte

In [ ]:
propre = df.rename(columns={"bill_length_mm": "bec_mm", "body_mass_g": "masse_g"})
propre["sex"] = propre["sex"].str.lower()            # méthodes texte avec .str
propre["species"] = propre["species"].astype("category")
propre.dtypes

## Les dates

In [ ]:
ventes = pd.DataFrame({"date": ["2026-01-05", "2026-02-17", "2026-02-28"], "montant": [120, 80, 200]})
ventes["date"] = pd.to_datetime(ventes["date"])
ventes["mois"] = ventes["date"].dt.month
ventes

> **À retenir** : ne modifie jamais tes données brutes. Travaille sur une copie (`df.copy()`) et note chaque choix de nettoyage : tu devras le justifier dans tes projets.

### Exercice 1 · Nettoyer penguins

À partir de `df`, crée `propre` : supprime les lignes où `sex` manque, puis remplace les valeurs manquantes restantes de chaque colonne numérique par la **médiane** de cette colonne. Il ne doit plus rester aucune valeur manquante.

<details><summary>Indice</summary>

dropna(subset=["sex"]), puis une boucle sur les colonnes numériques avec fillna(...median()).
</details>

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert propre.isna().sum().sum() == 0, "Il reste des valeurs manquantes."
assert len(propre) == 333, f"propre a {len(propre)} lignes, j'en attendais 333 (on retire seulement les lignes sans sex)."
print("Bravo, tout est correct.")

### Exercice 2 · Uniformiser du texte

Les villes de `clients` ont été saisies n'importe comment. Crée la colonne `ville_propre` : sans espaces au début et à la fin, avec une majuscule en début de mot (`.str.strip()` puis `.str.title()`). Puis crée `n_villes`, le nombre de villes distinctes après nettoyage.

<details><summary>Indice</summary>

Enchaîne les méthodes : .str.strip().str.title(). nunique() compte les valeurs distinctes.
</details>

In [ ]:
import pandas as pd
clients = pd.DataFrame({"ville": ["paris", " Lyon", "PARIS ", "lyon", "Marseille", "marseille  "]})

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(clients["ville_propre"]) == ["Paris", "Lyon", "Paris", "Lyon", "Marseille", "Marseille"], f"ville_propre vaut {list(clients['ville_propre'])}."
assert n_villes == 3, f"n_villes vaut {n_villes}."
print("Bravo, tout est correct.")

---
## Transformer : colonnes, groupby, pivot, merge

On utilise ici `tips.csv` : 244 additions d'un restaurant, avec le montant (`total_bill`), le pourboire (`tip`), le jour, le service (`Lunch`, `Dinner`), la taille de la table (`size`).

## Créer des colonnes

In [ ]:
import pandas as pd
tips = pd.read_csv("tips.csv")
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
tips["grande_table"] = tips["size"] >= 4
tips["service_fr"] = tips["time"].map({"Lunch": "Déjeuner", "Dinner": "Dîner"})
tips.head()

`pd.cut` découpe une variable numérique en tranches :

In [ ]:
tips["tranche"] = pd.cut(tips["total_bill"], bins=[0, 15, 30, 60], labels=["< 15 $", "15–30 $", "> 30 $"])
tips["tranche"].value_counts()

## groupby : agréger par groupe

C'est l'opération la plus utile de pandas : **découper** en groupes, **appliquer** un calcul, **combiner** les résultats.

*Animation interactive « groupby » : à voir sur la plateforme web.*

In [ ]:
tips.groupby("day")["tip_pct"].mean().round(1)

In [ ]:
tips.groupby(["day", "time"]).agg(
    nb=("tip", "size"),
    pourboire_moyen=("tip", "mean"),
    addition_max=("total_bill", "max"),
).round(2)

## pivot_table : un tableau croisé

In [ ]:
pd.pivot_table(tips, values="tip_pct", index="day", columns="smoker", aggfunc="mean").round(1)

## merge : joindre deux tables

Comme une jointure SQL : on associe les lignes de deux tables qui partagent une clé.

In [ ]:
jours = pd.DataFrame({"day": ["Thur", "Fri", "Sat", "Sun"],
                      "jour": ["Jeudi", "Vendredi", "Samedi", "Dimanche"],
                      "week_end": [False, False, True, True]})
fusion = tips.merge(jours, on="day", how="left")
fusion.groupby("week_end")["tip_pct"].mean().round(2)

`how="left"` garde toutes les lignes de la table de gauche ; `inner` ne garde que les correspondances ; `outer` garde tout.

> **À retenir** : `groupby` + `agg` répond à la plupart des questions métier du type « quel est le X moyen par Y ? ». Entraîne-toi à traduire une question en phrase pandas.

### Exercice 1 · Pourboire moyen par jour

Ajoute à `tips` la colonne `tip_pct` (pourboire en % de l'addition). Crée la Series `par_jour` : le `tip_pct` moyen par jour (`day`), triée par ordre **décroissant**. Puis `meilleur_jour`, le jour au pourboire moyen le plus élevé.

<details><summary>Indice</summary>

groupby("day")["tip_pct"].mean() puis .sort_values(ascending=False). Le premier index est le meilleur jour.
</details>

In [ ]:
import pandas as pd
tips = pd.read_csv("tips.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
assert "tip_pct" in tips.columns, "Il manque la colonne tip_pct."
assert list(par_jour.index) == ["Fri", "Sun", "Thur", "Sat"], f"Ordre obtenu : {list(par_jour.index)}."
assert meilleur_jour == "Fri", f"meilleur_jour vaut {meilleur_jour!r}."
assert abs(par_jour["Fri"] - 16.99) < 0.01, f"Pour Fri, j'attendais environ 16.99 %, tu as {par_jour['Fri']:.2f}."
print("Bravo, tout est correct.")

### Exercice 2 · Tableau de synthèse

Crée `synthese` avec un `groupby` sur `time` et `agg` : une colonne `nb` (nombre d'additions), `addition_moy` (moyenne de `total_bill`) et `table_moy` (moyenne de `size`).

<details><summary>Indice</summary>

Syntaxe : .agg(nom_colonne=("colonne_source", "fonction")).
</details>

In [ ]:
import pandas as pd
tips = pd.read_csv("tips.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert set(["nb", "addition_moy", "table_moy"]) <= set(synthese.columns), f"Colonnes : {list(synthese.columns)}."
assert synthese.loc["Dinner", "nb"] == 176 and synthese.loc["Lunch", "nb"] == 68, "Les effectifs par service ne sont pas les bons."
assert abs(synthese.loc["Dinner", "addition_moy"] - 20.797) < 0.01, "addition_moy pour Dinner devrait valoir environ 20.80."
print("Bravo, tout est correct.")

### Exercice 3 · Jointure

Joins `commandes` et `produits` sur `produit_id` pour obtenir `fusion`, puis calcule `ca_par_categorie` : la somme de `quantite * prix` par `categorie`.

<details><summary>Indice</summary>

commandes.merge(produits, on="produit_id"), puis une colonne ca, puis groupby.
</details>

In [ ]:
import pandas as pd
commandes = pd.DataFrame({"produit_id": [1, 2, 1, 3, 2], "quantite": [2, 1, 5, 3, 2]})
produits = pd.DataFrame({"produit_id": [1, 2, 3], "categorie": ["livre", "cours", "livre"], "prix": [30, 200, 45]})

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert len(fusion) == 5, f"fusion a {len(fusion)} lignes, j'en attendais 5."
assert dict(ca_par_categorie) == {"cours": 600, "livre": 345}, f"ca_par_categorie vaut {dict(ca_par_categorie)}."
print("Bravo, tout est correct.")

---
## Visualiser avec Plotly Express

Chez Jedha, les graphiques se font avec **Plotly Express** : une ligne de code par graphique, et des graphiques interactifs (survol, zoom). Ici, tu disposes d'une **version simplifiée** qui accepte la même syntaxe pour `scatter`, `histogram`, `box`, `bar` et `line`. Sur ton ordinateur, le même code fonctionne avec la vraie bibliothèque.

## Choisir le bon graphique

- **Distribution** d'une variable numérique → `histogram` ou `box`.
- **Relation** entre deux variables numériques → `scatter`.
- **Comparaison** d'une valeur entre catégories → `bar`.
- **Évolution** dans le temps ou au fil des itérations → `line`.

## Histogramme

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("penguins.csv")
fig = px.histogram(df, x="body_mass_g", nbins=30, title="Distribution de la masse des manchots")
fig.show()

Ajoute `color=` pour séparer les groupes :

In [ ]:
px.histogram(df, x="flipper_length_mm", color="species", barmode="overlay",
             title="Longueur des nageoires par espèce")

## Nuage de points

In [ ]:
px.scatter(df, x="flipper_length_mm", y="body_mass_g", color="species",
           labels={"flipper_length_mm": "Nageoire (mm)", "body_mass_g": "Masse (g)"},
           title="Plus la nageoire est longue, plus le manchot est lourd")

## Boîte à moustaches

Elle montre la médiane, les quartiles et les valeurs extrêmes : idéale pour comparer des distributions.

In [ ]:
px.box(df, x="species", y="bill_depth_mm", title="Épaisseur du bec par espèce")

## Diagramme en barres

On agrège d'abord avec pandas, puis on trace.

In [ ]:
moy = df.groupby("species", as_index=False)["body_mass_g"].mean()
px.bar(moy, x="species", y="body_mass_g", title="Masse moyenne par espèce (g)")

> **À retenir** : un bon titre de graphique énonce la **conclusion** (« Les Gentoo sont les plus lourds »), pas seulement le contenu (« Masse par espèce »). Nomme toujours tes axes avec leurs unités.

### Exercice 1 · Pourboire contre addition

Avec `tips.csv`, crée une figure `fig` : un nuage de points du pourboire (`tip`, en y) en fonction de l'addition (`total_bill`, en x), coloré par `time`, avec un titre. Affiche-la.

<details><summary>Indice</summary>

px.scatter(tips, x=..., y=..., color=..., title=...)
</details>

In [ ]:
import pandas as pd
import plotly.express as px
tips = pd.read_csv("tips.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert getattr(fig, "_is_px_fig", False) or hasattr(fig, "data"), "fig doit être une figure Plotly Express."
assert fig.data[0]["type"] == "scatter", "Il faut un nuage de points (px.scatter)."
assert len(fig.data) == 2, "Colore par time : il doit y avoir 2 groupes (Lunch et Dinner)."
assert fig.layout.get("title"), "Ajoute un titre avec title=..."
print("Bravo, tout est correct.")

### Exercice 2 · Taux de survie par classe

Avec `titanic.csv`, calcule le taux de survie moyen par `pclass` dans un DataFrame `taux` (avec `as_index=False`), puis trace `fig`, un diagramme en barres de ce taux par classe.

<details><summary>Indice</summary>

La moyenne d'une colonne 0/1 est un taux. Puis px.bar(taux, x="pclass", y="survived").
</details>

In [ ]:
import pandas as pd
import plotly.express as px
titanic = pd.read_csv("titanic.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert len(taux) == 3, "taux doit avoir une ligne par classe."
assert fig.data[0]["type"] == "bar", "Il faut un diagramme en barres (px.bar)."
assert abs(max(fig.data[0]["y"]) - 0.6296) < 0.001, "Les hauteurs des barres ne correspondent pas aux taux de survie."
print("Bravo, tout est correct.")

---
## Repérer anomalies et biais

Avant de modéliser, il faut se demander si les données sont **fiables** et **représentatives**. Un modèle reproduit fidèlement les défauts des données sur lesquelles il apprend.

## Les valeurs aberrantes (outliers)

La méthode de l'**écart interquartile** (IQR) est la plus courante : une valeur est suspecte si elle s'éloigne de plus de 1,5 IQR des quartiles.

In [ ]:
import pandas as pd
tips = pd.read_csv("tips.csv")
q1, q3 = tips["total_bill"].quantile([0.25, 0.75])
iqr = q3 - q1
bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
aberrants = tips[(tips["total_bill"] < bas) | (tips["total_bill"] > haut)]
print(f"Bornes : [{bas:.2f} ; {haut:.2f}] → {len(aberrants)} valeurs suspectes")

Une valeur aberrante n'est pas forcément une erreur : une grosse addition est possible. On la vérifie, puis on décide de la garder, de la corriger ou de l'exclure, et on documente ce choix.

## Le z-score

Le z-score mesure à combien d'écarts-types une valeur se trouve de la moyenne. Au-delà de 3 en valeur absolue, elle est rare pour une distribution normale.

In [ ]:
z = (tips["tip"] - tips["tip"].mean()) / tips["tip"].std()
tips[z.abs() > 3][["total_bill", "tip"]]

## Le déséquilibre des classes

In [ ]:
titanic = pd.read_csv("titanic.csv")
titanic["survived"].value_counts(normalize=True).round(3)

Si 95 % des transactions sont légitimes, un modèle qui répond toujours « légitime » a 95 % d'exactitude, tout en étant inutile. Il faut le savoir **avant** d'évaluer un modèle (semaine 4).

## Les biais à garder en tête

- **Biais d'échantillonnage** : les données ne représentent pas la population visée (par exemple, un sondage fait uniquement en ligne).
- **Biais historique** : les données reflètent des décisions passées injustes, que le modèle va reproduire.
- **Fuite de données** : une colonne contient indirectement la réponse. Dans `titanic.csv`, la colonne `alive` est la même information que `survived` : l'utiliser pour prédire la survie serait tricher.
- **Corrélation n'est pas causalité** : deux variables peuvent évoluer ensemble à cause d'une troisième.

In [ ]:
pd.crosstab(titanic["survived"], titanic["alive"])

> **À retenir** : pour chaque jeu de données, pose-toi trois questions. Qui a collecté ces données, et comment ? Qui en est absent ? Quelle colonne ne sera pas disponible au moment de la prédiction ?

### Exercice 1 · Une fonction de détection

Écris `outliers_iqr(serie)` qui renvoie un **masque booléen** (une Series de True/False) indiquant les valeurs hors de `[Q1 - 1.5 IQR ; Q3 + 1.5 IQR]`. Puis crée `n_outliers`, le nombre de valeurs aberrantes de `tips["tip"]`.

<details><summary>Indice</summary>

Reprends le code de la leçon, mais renvoie le masque au lieu de filtrer.
</details>

In [ ]:
import pandas as pd
tips = pd.read_csv("tips.csv")

def outliers_iqr(serie):
    pass

Vérifie que l'affichage correspond exactement à la consigne.

### Exercice 2 · Repérer la fuite de données

Parmi les colonnes de `titanic.csv`, une colonne textuelle répète exactement l'information de `survived`. Mets son nom dans la variable `colonne_fuite`, puis crée `features_ok` : la liste des colonnes **sans** `survived` et sans cette colonne.

<details><summary>Indice</summary>

Regarde la colonne qui vaut yes/no. pd.crosstab permet de le confirmer.
</details>

In [ ]:
import pandas as pd
titanic = pd.read_csv("titanic.csv")
print(titanic.columns.tolist())

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert colonne_fuite == "alive", f"colonne_fuite vaut {colonne_fuite!r}."
assert "survived" not in features_ok and "alive" not in features_ok, "features_ok ne doit contenir ni survived ni la colonne de fuite."
assert len(features_ok) == 13, f"features_ok contient {len(features_ok)} colonnes, j'en attendais 13."
print("Bravo, tout est correct.")

---
## Statistiques descriptives

Les statistiques descriptives résument une variable en quelques nombres. Tu les utiliseras dans chaque analyse exploratoire, et elles servent à préparer les données d'un modèle (standardisation).

## Tendance centrale

- **Moyenne** : la somme divisée par le nombre de valeurs. Sensible aux valeurs extrêmes.
- **Médiane** : la valeur du milieu une fois les données triées. Robuste aux extrêmes.

In [ ]:
import numpy as np
salaires = np.array([2100, 2300, 2400, 2500, 2600, 15000])
print("moyenne :", salaires.mean())
print("médiane :", np.median(salaires))

Un seul salaire très élevé fait monter la moyenne bien au-dessus de ce que gagnent la plupart des gens. C'est pour cela qu'on publie souvent des salaires **médians**.

## Dispersion

La **variance** est la moyenne des carrés des écarts à la moyenne. L'**écart-type** est sa racine carrée, dans la même unité que les données.

$$
\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2
$$

In [ ]:
x = np.array([4.0, 8.0, 6.0, 2.0])
ecarts = x - x.mean()
variance = (ecarts ** 2).mean()
print(variance, np.var(x), np.sqrt(variance), np.std(x))

Attention : pandas calcule par défaut la variance **d'échantillon** (division par n − 1), NumPy la variance de population (division par n). L'écart est minime sur de grands jeux de données.

## Quantiles

Les **quartiles** coupent les données triées en quatre parts égales. Q1 = 25 %, Q2 = médiane, Q3 = 75 %.

In [ ]:
print(np.percentile(salaires, [25, 50, 75]))

## Corrélation

Le **coefficient de corrélation de Pearson** mesure la force d'une relation **linéaire** entre deux variables, de −1 (parfaitement opposées) à +1 (parfaitement liées). Proche de 0 : pas de relation linéaire.

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()
df[["bill_length_mm", "flipper_length_mm", "body_mass_g"]].corr().round(2)

> **À retenir** : une corrélation de 0,87 entre la nageoire et la masse dit que les deux augmentent ensemble. Elle ne dit pas que l'une cause l'autre.

### Exercice 1 · Variance à la main

Sans utiliser `np.var` ni `np.std`, calcule `variance` et `ecart_type` (version population, division par n) du tableau `x`, à partir de la formule.

<details><summary>Indice</summary>

Calcule la moyenne, les écarts au carré, leur moyenne, puis la racine carrée (** 0.5).
</details>

In [ ]:
import numpy as np
x = np.array([12.0, 15.0, 9.0, 14.0, 10.0])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert abs(variance - 5.2) < 1e-9, f"variance vaut {variance}, j'attendais 5.2."
assert abs(ecart_type - np.sqrt(5.2)) < 1e-9, f"ecart_type vaut {ecart_type}."
print("Bravo, tout est correct.")

### Exercice 2 · Quelle variable est la plus liée à la masse ?

Dans `penguins.csv` (lignes incomplètes supprimées), calcule `correlations` : la corrélation de chaque colonne numérique avec `body_mass_g`, **sans** `body_mass_g` elle-même. Puis `meilleure`, le nom de la colonne la plus corrélée.

<details><summary>Indice</summary>

df[colonnes].corr()["body_mass_g"] donne une Series ; .drop(...) retire une étiquette ; .idxmax() donne l'étiquette du maximum.
</details>

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert "body_mass_g" not in correlations.index, "Retire body_mass_g de correlations."
assert len(correlations) == 3, f"correlations doit contenir 3 valeurs, elle en a {len(correlations)}."
assert meilleure == "flipper_length_mm", f"meilleure vaut {meilleure!r}."
print("Bravo, tout est correct.")

---
## Probabilités et lois usuelles

Un modèle de classification ne dit pas « ce mail est un spam ». Il dit « ce mail a 93 % de probabilité d'être un spam ». Les probabilités sont le langage des modèles.

## Les bases

Une probabilité est un nombre entre 0 et 1. La **probabilité conditionnelle** P(A | B) se lit « probabilité de A sachant B » :

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
$$

In [ ]:
import pandas as pd
t = pd.read_csv("titanic.csv")
p_survie = t["survived"].mean()
p_survie_si_femme = t[t["sex"] == "female"]["survived"].mean()
print(f"P(survie) = {p_survie:.2f}   P(survie | femme) = {p_survie_si_femme:.2f}")

## Le théorème de Bayes

Il permet de « retourner » une probabilité conditionnelle :

$$
P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}
$$

Exemple classique : une maladie touche 1 % de la population. Un test la détecte dans 99 % des cas, mais se trompe aussi chez 5 % des personnes saines. Si ton test est positif, quelle est la probabilité que tu sois malade ?

In [ ]:
p_malade = 0.01
p_pos_si_malade = 0.99
p_pos_si_sain = 0.05
p_pos = p_pos_si_malade * p_malade + p_pos_si_sain * (1 - p_malade)
p_malade_si_pos = p_pos_si_malade * p_malade / p_pos
print(f"{p_malade_si_pos:.1%}")

Seulement 17 % ! La maladie est si rare que la plupart des tests positifs sont des faux positifs. C'est la même logique que pour les classes déséquilibrées en ML.

## Simuler pour comprendre

Quand un calcul semble difficile, on peut **simuler**. La **loi des grands nombres** dit que la fréquence observée se rapproche de la probabilité quand on répète l'expérience.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    lancers = rng.integers(0, 2, size=n)      # 0 = pile, 1 = face
    print(n, "lancers → fréquence de face :", lancers.mean())

## Deux lois à connaître

*Animation interactive « normal » : à voir sur la plateforme web.*

La **loi binomiale** compte les succès sur n essais indépendants de probabilité p. La **loi normale** (courbe en cloche) décrit beaucoup de mesures naturelles. Environ 68 % des valeurs sont à moins d'un écart-type de la moyenne, et 95 % à moins de deux.

In [ ]:
normale = rng.normal(loc=170, scale=10, size=100_000)   # tailles : moyenne 170, écart-type 10
print(np.mean(np.abs(normale - 170) < 10))               # environ 0.68
print(np.mean(np.abs(normale - 170) < 20))               # environ 0.95
binom = rng.binomial(n=10, p=0.3, size=100_000)
print(binom.mean())                                       # environ n × p = 3

> **À retenir** : la loi normale est partout en ML : initialisation des réseaux de neurones, hypothèses de la régression linéaire, standardisation des données.

### Exercice 1 · Une fonction Bayes

Écris `bayes(prior, sensibilite, faux_positifs)` qui renvoie la probabilité d'être malade sachant un test positif. `prior` = P(malade), `sensibilite` = P(positif | malade), `faux_positifs` = P(positif | sain).

<details><summary>Indice</summary>

Calcule d'abord P(positif) en additionnant les deux façons d'être positif.
</details>

In [ ]:
def bayes(prior, sensibilite, faux_positifs):
    pass

print(bayes(0.01, 0.99, 0.05))

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(bayes(0.01, 0.99, 0.05) - 0.16667) < 1e-4, f"bayes(0.01, 0.99, 0.05) renvoie {bayes(0.01, 0.99, 0.05)}, j'attendais environ 0.1667."
assert abs(bayes(0.5, 0.9, 0.1) - 0.9) < 1e-9, "Avec prior=0.5, sensibilite=0.9, faux_positifs=0.1, j'attends 0.9."
print("Bravo, tout est correct.")

### Exercice 2 · Simuler deux dés

Avec `rng`, simule 100 000 lancers de **deux** dés à six faces. Calcule `p_sept`, la fréquence à laquelle leur somme vaut 7. La valeur théorique est 1/6.

<details><summary>Indice</summary>

rng.integers(1, 7, size=n) tire des entiers de 1 à 6. Additionne les deux tableaux, compare à 7, prends la moyenne.
</details>

In [ ]:
import numpy as np
rng = np.random.default_rng(1)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(p_sept - 1 / 6) < 0.01, f"p_sept vaut {p_sept}, loin de 1/6 ≈ 0.1667. Vérifie les bornes de integers (la fin est exclue)."
print("Bravo, tout est correct.")

---
## Projet 2 · Analyse exploratoire du Titanic

Objectif : produire un notebook d'analyse complet, comme ceux que tu rendras chez Jedha. Pose des questions, réponds avec pandas et des graphiques, et conclus.

Les exercices ci-dessous te guident sur les étapes clés. Reproduis-les ensuite dans un notebook Jupyter sur ton ordinateur, en ajoutant entre chaque étape une cellule de texte qui explique ce que tu observes.

**À faire :**

- [ ] Refaire l'analyse dans un notebook Jupyter sur ton ordinateur
- [ ] Poser 5 questions et y répondre, chacune avec un tableau ou un graphique
- [ ] Écrire une conclusion de 5 lignes maximum
- [ ] Publier le notebook et un README sur GitHub

### Exercice 1 · Audit de qualité

Charge `titanic.csv` dans `df`. Crée `manquants` : le **pourcentage** de valeurs manquantes par colonne, trié par ordre décroissant, en ne gardant que les colonnes qui en ont.

<details><summary>Indice</summary>

df.isna().mean() donne la proportion par colonne. Multiplie par 100, trie, puis filtre > 0.
</details>

In [ ]:
import pandas as pd

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(manquants.index[:2]) == ["deck", "age"], f"Les colonnes les plus incomplètes devraient être deck puis age, tu as {list(manquants.index[:2])}."
assert abs(manquants.iloc[0] - 77.2) < 0.1, f"deck devrait avoir environ 77.2 % de valeurs manquantes."
assert (manquants > 0).all(), "Ne garde que les colonnes qui ont des valeurs manquantes."
print("Bravo, tout est correct.")

### Exercice 2 · Survie par sexe et par classe

Crée `tableau` : un `pivot_table` du taux de survie (`survived`) avec `sex` en lignes et `pclass` en colonnes.

<details><summary>Indice</summary>

pd.pivot_table(df, values=..., index=..., columns=..., aggfunc="mean")
</details>

In [ ]:
import pandas as pd
df = pd.read_csv("titanic.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert tableau.shape == (2, 3), f"tableau a la forme {tableau.shape}, j'attendais (2, 3)."
assert abs(tableau.loc["female", 1] - 0.968) < 0.001, "Le taux de survie des femmes de 1re classe devrait être d'environ 0.968."
assert abs(tableau.loc["male", 3] - 0.135) < 0.001, "Le taux de survie des hommes de 3e classe devrait être d'environ 0.135."
print("Bravo, tout est correct.")

### Exercice 3 · L'âge des survivants

Crée `fig`, un histogramme de `age` coloré par `alive`, avec `barmode="overlay"` et un titre qui énonce ce que tu observes. Puis crée `taux_enfants`, le taux de survie des passagers de moins de 12 ans.

<details><summary>Indice</summary>

px.histogram(df, x="age", color="alive", barmode="overlay"). Puis filtre df["age"] < 12.
</details>

In [ ]:
import pandas as pd
import plotly.express as px
df = pd.read_csv("titanic.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert fig.data[0]["type"] == "histogram", "fig doit être un histogramme."
assert len(fig.data) == 2, "Colore par alive : 2 groupes attendus."
assert abs(taux_enfants - 0.5735) < 0.001, f"taux_enfants vaut {taux_enfants}, j'attendais environ 0.574."
print("Bravo, tout est correct.")